# baseline model tests (topography)

This notebook trains quick text-based baselines on extracted CAD metadata.
It uses **Layer / Type / Text** only (no geometry).

What we evaluate:
- Train/Val/Test split by `Source_File` (prevents leakage).
- Macro-F1 + Weighted-F1 + Accuracy for `l1` and `l2`.
- Optional GroupKFold cross-validation on the train split.

Notes:
- The regions dataset is large; start with a sample, then scale up.
- Test split should be used only once for final reporting.


In [11]:
# If scikit-learn is missing, uncomment:
# !pip install -q scikit-learn


In [12]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)
rand_seed = 44


In [13]:
# Paths
base = Path(r'C:/Users/artem/MAIN/projects/plain/local/python_modules/scripts/geo')
msk_path = base / 'labeled_msk_df.csv'
regions_path = base / 'labeled_regions_df.csv'

# Sampling for quick tests 
msk_samples_num = 200_000
regions_samples_num = 200_000

# Split settings
test_size = 0.2
val_size = 0.1  # fraction of remaining train after test split

# Cross-validation on train split
apply_cross_val = False
cross_val_splits = 3


In [ ]:
usecols = ['Source_File', 'Layer', 'Type', 'Text', 'l1', 'l2']

def load_df(path: Path, sample_n=None):
    print(f'Loading: {path}')
    df = pd.read_csv(path, usecols=lambda c: c in usecols, low_memory=False)

    # Normalize columns
    df.columns = [c.strip() for c in df.columns]

    # Ensure required
    if 'Source_File' not in df.columns:
        raise ValueError('Column Source_File is required')

    # Pick label columns
    label_l1 = 'l1' if 'l1' in df.columns else 'l1_auto'
    label_l2 = 'l2' if 'l2' in df.columns else 'l2_auto'

    # Basic cleanup
    for col in ['Layer', 'Type', 'Text']:
        if col not in df.columns:
            df[col] = ''
        df[col] = df[col].fillna('').astype(str)

    df[label_l1] = df[label_l1].fillna('')
    df[label_l2] = df[label_l2].fillna('')

    # Optional sampling
    if sample_n is not None and len(df) > sample_n:
        df = df.sample(n=sample_n, random_state=rand_seed)
        df = df.reset_index(drop=True)

    print('Columns:', list(df.columns))
    print('Rows:', len(df))
    print('Label columns:', label_l1, label_l2)
    return df, label_l1, label_l2


In [15]:
def build_text_features(df: pd.DataFrame):
    # Simple concatenation of fields into a single text input
    return (
        'TYPE=' + df['Type'].astype(str) + ' ' +
        'LAYER=' + df['Layer'].astype(str) + ' ' +
        'TEXT=' + df['Text'].astype(str)
    )


In [16]:
def split_train_val_test(X, y, groups):
    # 1) Test split
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=rand_seed)
    train_val_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train_val = X.iloc[train_val_idx]
    y_train_val = y.iloc[train_val_idx]
    groups_train_val = groups.iloc[train_val_idx]

    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]
    groups_test = groups.iloc[test_idx]

    # 2) Val split from train_val
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=rand_seed)
    train_idx, val_idx = next(gss2.split(X_train_val, y_train_val, groups=groups_train_val))

    X_train = X_train_val.iloc[train_idx]
    y_train = y_train_val.iloc[train_idx]
    groups_train = groups_train_val.iloc[train_idx]

    X_val = X_train_val.iloc[val_idx]
    y_val = y_train_val.iloc[val_idx]
    groups_val = groups_train_val.iloc[val_idx]

    return X_train, y_train, groups_train, X_val, y_val, groups_val, X_test, y_test, groups_test


In [17]:
def build_model():
    # class_weight='balanced' helps with class imbalance
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=2,
            max_features=200_000,
        )),
        ('clf', SGDClassifier(
            loss='log_loss',
            alpha=1e-5,
            max_iter=5,
            n_jobs=-1,
            random_state=rand_seed,
            class_weight='balanced'
        ))
    ])


In [18]:
def eval_split(y_true, y_pred, split_name):
    macro = f1_score(y_true, y_pred, average='macro')
    weighted = f1_score(y_true, y_pred, average='weighted')
    acc = accuracy_score(y_true, y_pred)
    print(f'{split_name:<6} | Macro-F1: {macro:.4f} | Weighted-F1: {weighted:.4f} | Acc: {acc:.4f}')

def train_eval(df: pd.DataFrame, label_col: str, name: str):
    # Filter rows with labels
    y = df[label_col].astype(str)
    mask = y != ''
    df = df[mask].copy()
    y = y[mask]

    # Build features
    X_text = build_text_features(df)
    groups = df['Source_File']

    X_train, y_train, groups_train, X_val, y_val, groups_val, X_test, y_test, groups_test = split_train_val_test(
        X_text, y, groups
    )

    model = build_model()
    model.fit(X_train, y_train)

    print(f'\n==== {name} | {label_col} ====')
    print(f'Classes: {y.nunique()}')
    print(f'Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}')

    eval_split(y_train, model.predict(X_train), 'train')
    eval_split(y_val,   model.predict(X_val),   'val')
    eval_split(y_test,  model.predict(X_test),  'test')

    # Optional CV on train split only
    if apply_cross_val:
        print('\nRunning GroupKFold CV on train split...')
        gkf = GroupKFold(n_splits=cross_val_splits)
        cv_macros = []
        for i, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups_train), 1):
            m = build_model()
            m.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
            preds = m.predict(X_train.iloc[va_idx])
            macro = f1_score(y_train.iloc[va_idx], preds, average='macro')
            cv_macros.append(macro)
            print(f'  Fold {i}: Macro-F1 = {macro:.4f}')
        print(f'CV Macro-F1 mean: {np.mean(cv_macros):.4f} | std: {np.std(cv_macros):.4f}')

    return model


In [19]:
# Moscow baseline
msk_df, msk_l1, msk_l2 = load_df(msk_path, sample_n=msk_samples_num)

model_msk_l1 = train_eval(msk_df, msk_l1, 'Moscow')
model_msk_l2 = train_eval(msk_df, msk_l2, 'Moscow')


Loading: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\labeled_msk_df.csv
Columns: ['Source_File', 'Layer', 'Type', 'Text', 'l1', 'l2']
Rows: 180953
Label columns: l1 l2


c:\ProgramData\anaconda3\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:702: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(



==== Moscow | l1 ====
Classes: 12
Train size: 104953 | Val size: 5886 | Test size: 70114
train  | Macro-F1: 0.9642 | Weighted-F1: 0.9978 | Acc: 0.9978
val    | Macro-F1: 0.9034 | Weighted-F1: 0.9963 | Acc: 0.9961
test   | Macro-F1: 0.9238 | Weighted-F1: 0.9919 | Acc: 0.9912


c:\ProgramData\anaconda3\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:702: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(



==== Moscow | l2 ====
Classes: 31
Train size: 87613 | Val size: 4807 | Test size: 65982
train  | Macro-F1: 0.9551 | Weighted-F1: 0.9970 | Acc: 0.9968
val    | Macro-F1: 0.8823 | Weighted-F1: 0.9970 | Acc: 0.9969
test   | Macro-F1: 0.9122 | Weighted-F1: 0.9898 | Acc: 0.9892


In [20]:
# Regions baseline
reg_df, reg_l1, reg_l2 = load_df(regions_path, sample_n=regions_samples_num)

model_reg_l1 = train_eval(reg_df, reg_l1, 'Regions')
model_reg_l2 = train_eval(reg_df, reg_l2, 'Regions')


Loading: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\labeled_regions_df.csv
Columns: ['Source_File', 'Layer', 'Type', 'Text', 'l1', 'l2']
Rows: 200000
Label columns: l1 l2


c:\ProgramData\anaconda3\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:702: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(



==== Regions | l1 ====
Classes: 16
Train size: 155691 | Val size: 7133 | Test size: 37176
train  | Macro-F1: 0.7166 | Weighted-F1: 0.9538 | Acc: 0.9495
val    | Macro-F1: 0.5480 | Weighted-F1: 0.9011 | Acc: 0.8975
test   | Macro-F1: 0.6311 | Weighted-F1: 0.9445 | Acc: 0.9422


c:\ProgramData\anaconda3\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:702: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(



==== Regions | l2 ====
Classes: 72
Train size: 124418 | Val size: 6332 | Test size: 30568
train  | Macro-F1: 0.5513 | Weighted-F1: 0.9405 | Acc: 0.9337
val    | Macro-F1: 0.3831 | Weighted-F1: 0.8950 | Acc: 0.8895
test   | Macro-F1: 0.4683 | Weighted-F1: 0.9172 | Acc: 0.8999


## Next ideas
- Increase sample sizes or run full dataset if memory allows.
- Try word-level TF-IDF and/or add simple numeric features (length of text, layer length, etc.).
- Compare Moscow-trained model vs Regions test (domain shift).
